In [ ]:
pip install torch==2.8.0 torchaudio==2.8.0

In [14]:
import os
import tarfile
import random
import io
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T
from torchaudio.utils import download_asset
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import librosa
import whisper
from jiwer import wer
from tqdm.auto import tqdm
import IPython.display as ipd

In [15]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(42)

In [16]:
class Config:
    sr = 16000
    batch_size = 8
    epochs = 100
    lr = 1e-4
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tar_path = "ru_train_0_19.tar"
    tsv_path = "train(1).tsv"
    extract_dir = "./ru_train_data"
    
def clean_text(text):
    return re.sub(r'[^\w\s]', '', str(text).lower()).strip()

if not os.path.exists(Config.extract_dir):
    print("Распаковка архива...")
    os.makedirs(Config.extract_dir, exist_ok=True)
    with tarfile.open(Config.tar_path, "r") as tar:
        tar.extractall(path=Config.extract_dir)
    print("Распаковка завершена.")

df_train = pd.read_csv(Config.tsv_path, sep='\t')
reference_dict = {row['path']: clean_text(row['sentence']) for _, row in df_train.iterrows()}

In [17]:
def get_snr_scale(signal, noise, snr_db):
    sig_power = signal.norm(p=2)**2 / (signal.numel() + 1e-8)
    noise_power = noise.norm(p=2)**2 / (noise.numel() + 1e-8)
    target_noise_power = sig_power / (10 ** (snr_db / 10))
    return torch.sqrt(target_noise_power / (noise_power + 1e-8))

print("Загрузка ассетов шума...")
babble_path = download_asset("tutorial-assets/Lab41-SRI-VOiCES-rm1-babb-mc01-stu-clo-8000hz.wav")
BABBLE_WAVEFORM, sr_b = torchaudio.load(babble_path)
BABBLE_WAVEFORM = T.Resample(sr_b, Config.sr)(BABBLE_WAVEFORM.mean(dim=0, keepdim=True))

rir_path = download_asset("tutorial-assets/Lab41-SRI-VOiCES-rm1-impulse-mc01-stu-clo-8000hz.wav")
RIR_WAVEFORM, sr_r = torchaudio.load(rir_path)
RIR_WAVEFORM = T.Resample(sr_r, Config.sr)(RIR_WAVEFORM.mean(dim=0, keepdim=True))
RIR_WAVEFORM = RIR_WAVEFORM[:, :int(Config.sr * 0.3)] 
RIR_WAVEFORM = RIR_WAVEFORM / torch.norm(RIR_WAVEFORM, p=2)

Загрузка ассетов шума...


/tmp/ipykernel_67479/2890745719.py:8: UserWarning: torchaudio.utils.download.download_asset has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  babble_path = download_asset("tutorial-assets/Lab41-SRI-VOiCES-rm1-babb-mc01-stu-clo-8000hz.wav")
/opt/conda/lib/python3.11/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/opt/conda/lib/python3.11/site-packa

In [18]:
def apply_noise(clean, force_type=None, file_seed=None):
    if file_seed is not None:
        random.seed(file_seed)
        
    snr = random.uniform(-5, 15)
    n_len = clean.shape[-1]

    allowed_noises = ['babble', 'rir', 'white']

    if force_type is not None and force_type in allowed_noises:
        noise_cat = force_type
    else:
        noise_cat = random.choice(allowed_noises)

    if noise_cat == 'babble':
        noise = BABBLE_WAVEFORM
        if noise.shape[-1] < n_len:
            repeats = (n_len // noise.shape[-1]) + 2
            noise = noise.repeat(1, repeats)
        max_start = noise.shape[-1] - n_len
        start = random.randint(0, max_start) if max_start > 0 else 0
        noise_crop = noise[:, start:start+n_len]
        scale = get_snr_scale(clean, noise_crop, snr)
        noisy = clean + noise_crop * scale

    elif noise_cat == 'rir':
        rir = RIR_WAVEFORM
        n_fft_conv = n_len + rir.shape[-1] - 1
        clean_fft = torch.fft.rfft(clean, n=n_fft_conv)
        rir_fft = torch.fft.rfft(rir, n=n_fft_conv)
        augmented = torch.fft.irfft(clean_fft * rir_fft, n=n_fft_conv)
        noisy = augmented[:, :n_len]
        white = torch.randn_like(clean)
        scale = get_snr_scale(noisy, white, snr + 10)
        noisy = noisy + white * scale

    elif noise_cat == 'white':
        noise = torch.randn(1, n_len, device=clean.device)
        noise = noise / (noise.abs().max() + 1e-8)
        scale = get_snr_scale(clean, noise, snr)
        noisy = clean + noise * scale

    max_val = noisy.abs().max()
    if max_val > 1.0:
        noisy = noisy / (max_val + 1e-8)

    return noisy

In [19]:
class SpeechEnhancementDataset(Dataset):
    def __init__(self, data_dir, ref_dict, is_train=True, max_len_sec=3.0):
        self.data_dir = data_dir
        self.ref_dict = ref_dict
        self.is_train = is_train
        self.max_len_sec = max_len_sec
        self.files = []
        for root, _, files in os.walk(data_dir):
            for f in files:
                if f.endswith('.mp3') and f in ref_dict:
                    self.files.append(os.path.join(root, f))

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        file_path = self.files[idx]
        filename = os.path.basename(file_path)
        text = self.ref_dict[filename]

        waveform, sr = torchaudio.load(file_path)
        if sr != Config.sr:
            waveform = T.Resample(sr, Config.sr)(waveform)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        if self.is_train:
            max_samples = int(self.max_len_sec * Config.sr)
            if waveform.shape[-1] > max_samples:
                start = random.randint(0, waveform.shape[-1] - max_samples)
                waveform = waveform[:, start:start + max_samples]
            else:
                pad_len = max_samples - waveform.shape[-1]
                waveform = F.pad(waveform, (0, pad_len))
        
        clean = waveform
        noisy = apply_noise(clean) if self.is_train else clean

        return noisy.squeeze(0), clean.squeeze(0), text

def collate_fn(batch):
    noisy_list, clean_list, texts = [], [], []
    for noisy, clean, text in batch:
        noisy_list.append(noisy)
        clean_list.append(clean)
        texts.append(text)

    noisy_padded = pad_sequence(noisy_list, batch_first=True)
    clean_padded = pad_sequence(clean_list, batch_first=True)
    return noisy_padded, clean_padded, texts

dataset = SpeechEnhancementDataset(Config.extract_dir, reference_dict, is_train=True)
train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size

generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset = torch.utils.data.random_split(
    dataset, [train_size, val_size], generator=generator
)

train_loader = DataLoader(train_dataset, batch_size=Config.batch_size, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=Config.batch_size, shuffle=False, collate_fn=collate_fn)

In [20]:
import torch
import torch.nn as nn
from typing import Tuple, List

def complex_mul(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    real = a[..., 0] * b[..., 0] - a[..., 1] * b[..., 1]
    imag = a[..., 0] * b[..., 1] + a[..., 1] * b[..., 0]
    return torch.stack([real, imag], dim=-1)

def apply_complex_mask(X: torch.Tensor, M: torch.Tensor, R: torch.Tensor) -> torch.Tensor:
    return complex_mul(X, M) + R

class BandSplitModule(nn.Module):
    def __init__(self, band_bins: List[int], feature_dim: int):
        super().__init__()
        self.band_bins = band_bins
        self.num_bands = len(band_bins)
        
        self.norms = nn.ModuleList([nn.LayerNorm(b * 2) for b in band_bins])
        self.projs = nn.ModuleList([nn.Linear(b * 2, feature_dim) for b in band_bins])
        
    def forward(self, X: torch.Tensor) -> torch.Tensor:
        B, _, T, _ = X.shape
        
        X_splits = torch.split(X, self.band_bins, dim=1)
        
        Z_bands = []
        for i in range(self.num_bands):
            x_i = X_splits[i]
            x_i = x_i.permute(0, 2, 1, 3).contiguous()
            x_i = x_i.view(B, T, -1)
            
            z_i = self.norms[i](x_i)
            z_i = self.projs[i](z_i)
            
            Z_bands.append(z_i)
            
        Z = torch.stack(Z_bands, dim=1)
        return Z

class TemporalModel(nn.Module):
    def __init__(self, feature_dim: int, hidden_size: int, num_layers: int):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=feature_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )
        self.proj = nn.Linear(hidden_size, feature_dim)
        
    def forward(self, Z: torch.Tensor) -> torch.Tensor:
        B, num_bands, T, D = Z.shape
        
        Z_reshaped = Z.view(B * num_bands, T, D)
        
        out, _ = self.lstm(Z_reshaped)
        out = self.proj(out)
        out = out + Z_reshaped
        out = out.view(B, num_bands, T, D)
        return out

class BandModel(nn.Module):
    def __init__(self, feature_dim: int, hidden_size: int, num_layers: int, split_index: int = 26):
        super().__init__()
        self.split_index = split_index
        
        self.low_rnn = nn.LSTM(
            input_size=feature_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True
        )
        self.low_proj = nn.Linear(hidden_size * 2, feature_dim)
        
        self.high_rnn = nn.LSTM(
            input_size=feature_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=False
        )
        self.high_proj = nn.Linear(hidden_size, feature_dim)
        
    def forward(self, Z: torch.Tensor) -> torch.Tensor:
        B, num_bands, T, D = Z.shape
        Z_perm = Z.permute(0, 2, 1, 3).contiguous()
        Z_reshaped = Z_perm.view(B * T, num_bands, D)
        
        Z_low = Z_reshaped[:, :self.split_index, :]
        Z_high = Z_reshaped[:, self.split_index:, :]
        
        low_out, _ = self.low_rnn(Z_low)
        low_out = self.low_proj(low_out)
        low_out = low_out + Z_low
        
        if Z_high.size(1) > 0:
            high_out, _ = self.high_rnn(Z_high)
            high_out = self.high_proj(high_out)
            high_out = high_out + Z_high
            out_reshaped = torch.cat([low_out, high_out], dim=1)
        else:
            out_reshaped = low_out
        
        out = out_reshaped.view(B, T, num_bands, D)
        out = out.permute(0, 2, 1, 3).contiguous()
        
        return out

class MaskEstimator(nn.Module):
    def __init__(self, band_bins: List[int], feature_dim: int):
        super().__init__()
        self.num_bands = len(band_bins)
        
        self.mlps = nn.ModuleList()
        for b in band_bins:
            self.mlps.append(nn.Sequential(
                nn.Linear(feature_dim, 384),
                nn.Tanh(),
                nn.Linear(384, b * 8),
                nn.GLU(dim=-1)
            ))
            
    def forward(self, Z: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        B, num_bands, T, D = Z.shape
        
        out_bands = []
        for i in range(self.num_bands):
            z_i = Z[:, i, :, :]
            out_i = self.mlps[i](z_i)
            
            b_i = out_i.shape[-1] // 4
            out_i = out_i.view(B, T, b_i, 4)
            
            out_i = out_i.permute(0, 2, 1, 3).contiguous()
            out_bands.append(out_i)
            
        out = torch.cat(out_bands, dim=1)
        
        M = out[..., :2]
        R = out[..., 2:]
        
        return M, R

class BSRNN(nn.Module):
    def __init__(self, freq_bins: int, feature_dim: int, hidden_size: int, num_layers: int):
        super().__init__()
        
        target_bandwidths = [200] * 20 + [500] * 6 + [2000] * 7
        total_bandwidth = sum(target_bandwidths)
        
        self.band_bins = []
        remaining_bins = freq_bins
        for bw in target_bandwidths[:-1]:
            b = max(1, int(freq_bins * (bw / total_bandwidth)))
            self.band_bins.append(b)
            remaining_bins -= b
        self.band_bins.append(remaining_bins)
        
        self.band_split = BandSplitModule(self.band_bins, feature_dim)
        self.temporal_model = TemporalModel(feature_dim, hidden_size, num_layers)
        
        self.band_model = BandModel(feature_dim, hidden_size, num_layers, split_index=26)
        
        self.mask_estimator = MaskEstimator(self.band_bins, feature_dim)
        
    def forward(self, X: torch.Tensor) -> torch.Tensor:

        Z = self.band_split(X)
        Z = self.temporal_model(Z)
        Z = self.band_model(Z)
        M, R = self.mask_estimator(Z)
        S_hat = apply_complex_mask(X, M, R)
        S_hat = 0.8 * S_hat + 0.2 * X
        return S_hat

In [21]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class STFTLoss(nn.Module):
    def __init__(self, n_fft, hop_length, win_length):
        super(STFTLoss, self).__init__()
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.win_length = win_length
        self.window = nn.Parameter(torch.hann_window(win_length), requires_grad=False)

    def forward(self, x, y):
        x_stft = torch.stft(x, self.n_fft, self.hop_length, self.win_length, self.window, return_complex=True)
        y_stft = torch.stft(y, self.n_fft, self.hop_length, self.win_length, self.window, return_complex=True)

        x_mag = torch.abs(x_stft) + 1e-8
        y_mag = torch.abs(y_stft) + 1e-8

        sc_loss = torch.norm(y_mag - x_mag, p="fro") / torch.norm(y_mag, p="fro")
        
        mag_loss = F.l1_loss(torch.log(y_mag), torch.log(x_mag))

        return sc_loss + mag_loss

class MultiResolutionSTFTLoss(nn.Module):
    def __init__(self, fft_sizes=[512, 1024, 2048], hop_sizes=[120, 240, 480], win_lengths=[600, 1200, 2400]):
        super(MultiResolutionSTFTLoss, self).__init__()
        self.loss_layers = nn.ModuleList([
            STFTLoss(fs, hs, wl) for fs, hs, wl in zip(fft_sizes, hop_sizes, win_lengths)
        ])

    def forward(self, x, y):
        mr_stft_loss = 0
        for layer in self.loss_layers:
            mr_stft_loss += layer(x, y)
        return mr_stft_loss / len(self.loss_layers)

class CombinedLoss(nn.Module):
    def __init__(self, alpha=0.5):
        super(CombinedLoss, self).__init__()
        self.alpha = alpha
        self.mr_stft = MultiResolutionSTFTLoss(
            fft_sizes=[512, 1024, 2048],
            hop_sizes=[128, 256, 512],
            win_lengths=[512, 1024, 2048]
        )
        
    def forward(self, denoised, clean):

        target = clean - torch.mean(clean, dim=-1, keepdim=True)
        estimated = denoised - torch.mean(denoised, dim=-1, keepdim=True)
        
        dot = torch.sum(target * estimated, dim=-1, keepdim=True)
        norm = torch.sum(target ** 2, dim=-1, keepdim=True) + 1e-8
        
        target_scaled = (dot / norm) * target
        noise = estimated - target_scaled
        
        si_sdr_loss = -10 * torch.log10(
            (torch.sum(target_scaled ** 2, dim=-1) + 1e-8) / 
            (torch.sum(noise ** 2, dim=-1) + 1e-8)
        )
        si_sdr_loss = si_sdr_loss.mean()

        stft_loss = self.mr_stft(denoised, clean)

        return (1 - self.alpha) * si_sdr_loss + self.alpha * stft_loss

In [16]:
model = BSRNN(freq_bins=257, feature_dim=128, hidden_size=256, num_layers=2).to(Config.device)
criterion = CombinedLoss(alpha=0.8).to(Config.device)
optimizer = torch.optim.Adam(model.parameters(), lr=Config.lr)
n_fft = 512
hop_length = 256

for epoch in range(1, Config.epochs + 1):
    model.train()
    total_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{Config.epochs}")
    for noisy, clean, _ in pbar:
        noisy, clean = noisy.to(Config.device), clean.to(Config.device)
        optimizer.zero_grad()

        X_complex = torch.stft(noisy, n_fft=n_fft, hop_length=hop_length, return_complex=False)
        
        S_hat = model(X_complex) 
        
        S_hat_complex = torch.view_as_complex(S_hat)
        denoised = torch.istft(S_hat_complex, n_fft=n_fft, hop_length=hop_length, length=noisy.shape[-1])
        
        loss = criterion(denoised, clean)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        pbar.set_postfix({"Loss": f"{loss.item():.4f}"})
        
    print(f"Epoch {epoch} | Avg Loss (SI-SDR): {total_loss / len(train_loader):.4f}")

Epoch 1/100:   0%|          | 0/2977 [00:00<?, ?it/s]

/opt/conda/lib/python3.11/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be r

KeyboardInterrupt: 

In [59]:
torch.save(model.state_dict(), 'bsrnn_weights_final.pth')

In [10]:
model = BSRNN(freq_bins=257, feature_dim=128, hidden_size=256, num_layers=2).to(Config.device)
criterion = CombinedLoss(alpha=0.8).to(Config.device)
optimizer = torch.optim.Adam(model.parameters(), lr=Config.lr)
n_fft = 512
hop_length = 256

model_path = 'bsrnn_weights_final.pth'

if os.path.exists(model_path):
    # Загружаем state_dict
    state_dict = torch.load(model_path, map_location=Config.device)
    model.load_state_dict(state_dict)
    model.eval()
    print(f"Модель успешно загружена из {model_path}")
else:
    print(f"Файл {model_path} не найден!")

Модель успешно загружена из bsrnn_weights_final.pth


In [11]:
seed_everything(42)

In [22]:
def evaluate_and_listen(model, device, val_dataset, limit=None):
    asr = whisper.load_model("large-v3").to(device)
    model.eval()

    noise_types = ['babble', 'rir', 'white']
    stats = {n: {"wer_n": [], "wer_d": [], "sisdr_n": [], "sisdr_d": [], "samples": []} for n in noise_types}
    clean_wer = []

    if limit is not None:
        indices = list(range(min(limit, len(val_dataset))))
    else:
        indices = list(range(len(val_dataset)))

    with torch.no_grad():
        for idx in tqdm(indices, desc="Evaluation"):
            _, clean_wav, ref_text = val_dataset[idx]
            clean_np = clean_wav.numpy()

            t_clean = asr.transcribe(clean_np, fp16=False, language='ru')['text']
            clean_wer.append(wer(ref_text, clean_text(t_clean)))

            clean_tensor_cpu = clean_wav.unsqueeze(0)

            for n_type in noise_types:
                noisy_tensor_cpu = apply_noise(clean_tensor_cpu, force_type=n_type, file_seed=idx)
                noisy_tensor = noisy_tensor_cpu.to(device)
                X_comp = torch.stft(noisy_tensor.squeeze(1), n_fft=n_fft, hop_length=hop_length, return_complex=False)
                S_hat = model(X_comp)
                denoised_tensor = torch.istft(torch.view_as_complex(S_hat), n_fft=n_fft, hop_length=hop_length, length=noisy_tensor.shape[-1])

                noisy_np = noisy_tensor.squeeze().cpu().numpy()
                denoised_np = denoised_tensor.squeeze().cpu().numpy()

                t_n = asr.transcribe(noisy_np, fp16=False, language='ru')['text']
                t_d = asr.transcribe(denoised_np, fp16=False, language='ru')['text']

                def np_si_sdr(ref, est):
                    ref, est = ref.reshape(-1), est.reshape(-1)
                    dot = np.dot(ref, est)
                    norm = np.linalg.norm(ref)**2
                    proj = (dot / (norm + 1e-8)) * ref
                    noise = est - proj
                    return 10 * np.log10((np.linalg.norm(proj)**2) / (np.linalg.norm(noise)**2 + 1e-8) + 1e-8)

                stats[n_type]["wer_n"].append(wer(ref_text, clean_text(t_n)))
                stats[n_type]["wer_d"].append(wer(ref_text, clean_text(t_d)))
                stats[n_type]["sisdr_n"].append(np_si_sdr(clean_np, noisy_np))
                stats[n_type]["sisdr_d"].append(np_si_sdr(clean_np, denoised_np))

                if len(stats[n_type]["samples"]) < 3:
                    stats[n_type]["samples"].append({
                        "clean": clean_np, "noisy": noisy_np, "denoised": denoised_np,
                        "text": ref_text, "t_d": t_d
                    })

    print(f"\nBaseline WER (Clean Audio): {np.mean(clean_wer):.4f}")
    header = f"{'Noise Type':<10} | {'WER Noisy':<10} | {'WER Denois':<10} | {'Gain WER':<10} | {'SI-SDR Gain':<10}"
    print(header)

    for n_type in noise_types:
        w_n = np.mean(stats[n_type]["wer_n"])
        w_d = np.mean(stats[n_type]["wer_d"])
        s_n = np.mean(stats[n_type]["sisdr_n"])
        s_d = np.mean(stats[n_type]["sisdr_d"])
        print(f"{n_type:<10} | {w_n:<10.4f} | {w_d:<10.4f} | {w_n - w_d:<10.4f} | {s_d - s_n:<10.2f} dB")

    print("\nАУДИО ПРИМЕРЫ (по 3 на каждый шум):")
    for n_type in noise_types:
        print(f"ТИП ШУМА: {n_type.upper()}")
        for i, samp in enumerate(stats[n_type]["samples"]):
            print(f"\nПример {i+1}")
            print(f"Оригинальный текст: {samp['text']}")
            print(f"Распознано (Denoised): {samp['t_d']}")

            print("1. Чистое аудио:")
            ipd.display(ipd.Audio(samp["clean"], rate=Config.sr))

            print("2. Зашумленное аудио:")
            ipd.display(ipd.Audio(samp["noisy"], rate=Config.sr))

            print("3. Очищенное аудио:")
            ipd.display(ipd.Audio(samp["denoised"], rate=Config.sr))

evaluate_and_listen(model, Config.device, val_dataset, limit=20)

Evaluation:   0%|          | 0/20 [00:00<?, ?it/s]


Baseline WER (Clean Audio): 0.4156
Noise Type | WER Noisy  | WER Denois | Gain WER   | SI-SDR Gain
babble     | 0.5376     | 0.5416     | -0.0040    | 6.67       dB
rir        | 1.0000     | 1.0750     | -0.0750    | 30.58      dB
white      | 0.6243     | 0.7999     | -0.1756    | 7.99       dB

АУДИО ПРИМЕРЫ (по 3 на каждый шум):
ТИП ШУМА: BABBLE

Пример 1
Оригинальный текст: белым быком возрос над землей муууу
Распознано (Denoised):  Он возрос над землей.
1. Чистое аудио:


2. Зашумленное аудио:


3. Очищенное аудио:



Пример 2
Оригинальный текст: тем не менее он должен и будет идти вперед
Распознано (Denoised):  Он должен или будет идти вперед.
1. Чистое аудио:


...

2. Зашумленное аудио:


3. Очищенное аудио:



Пример 3
Оригинальный текст: конференция также должна делать свою работу в частности рассматривать свою нынешнюю повестку дня
Распознано (Denoised):  рассматривать свою нынешнюю поверхность дня.
1. Чистое аудио:


2. Зашумленное аудио:


3. Очищенное аудио:
